In [3]:
import arcpy
import math
import os

In [4]:
### ARCPY SPATIAL ANALYSIS after DOWNLOADING FINISH and GET THE DATA NEEDED ####
aprx = arcpy.mp.ArcGISProject("CURRENT")
map = aprx.listMaps()  # assumes data to be added to first map listed

In [5]:
map = [m for m in map if m.name == 'T4T_prioritize_planting'][0] #filtering the map name

IndexError: list index out of range

In [6]:
map = map[0]

In [7]:
map.name

'Map'

In [9]:
# workspace_scratch = r'Z:\GIS_ArcGISPro\TREEO'
# arcpy.env.workspace = workspace_scratch

In [11]:
arcpy.env.workspace

'C:\\Users\\q_bal\\Documents\\ArcGIS\\Projects\\yt_DEMO\\MyProject\\MyProject.gdb'

In [10]:
arcpy.ListWorkspaces("*","FileGDB")

[]

In [7]:
list_layers_name = [f.name for f in map.listLayers()]

In [8]:
list_layers_name

['grid_pattern_fishnet_3x4_filtered_Clip', 'point_line_per3m_contour_filtered_Clip', 'point_line_per3m_contourClip', 'grid_pattern_fishnet_3x4', 'point_line_per3m_contour', 'class_contour_filtered', 'main.contour_all_class', 'contour_line', 'prioritized_planting_filtered', 'Prioritized_planting', 'additional_t4t', 'Data_Blok_Petani_IPHPS_Mekarjaya', 'Plot_Delineated_merged', 't4t_tree_plot', 'measurement_approved_only', 'tree_space_buffered_1.5m', 'trees_buffered_gps_accuracy', 'contract_plot', 'measurement_buffered_tree_insideshp', 'Concession IPHPS - Social Forestry', 'DELINEATE_48S', 'slope_utm_fix.map', 'GEE', 'Slope_classes', 'FCD', 'Class_GEE_FCD_LC', 'PlanetLabs_used', 'T4T_202207_modified.tif', 'World Street Map']

In [9]:
contour_line = map.listLayers()[5]

In [10]:
print(contour_line.name) #check if the contour_line layer selected

class_contour_filtered


In [11]:
arcpy.Describe(contour_line).spatialReference

name (Projected Coordinate System),WGS_1984_UTM_Zone_48S
factoryCode (WKID),32748
linearUnitName (Linear Unit),Meter
name (Geographic Coordinate System),GCS_WGS_1984
factoryCode (WKID),4326
angularUnitName (Angular Unit),Degree
datumName (Datum),D_WGS_1984


In [12]:
arcpy.Describe(contour_line).SpatialReference.type

'Projected'

In [13]:
arcpy.Describe(contour_line).name

'class_contour_filtered'

In [14]:
# # This one actually if sr is projected
arcpy.env.overwriteOutput=True

interval = 3 #3 meter
lineClusTol = 1 # cluster tool, to make sure the last segment

# outDir = workspace_scratch

try:
    interval = 3  # Set your desired interval
    lineClusTol = 1  # Set your line clustering tolerance

    # Get the spatial reference of the contour_line feature class
    sr = arcpy.Describe(contour_line).spatialReference
    inLineName = arcpy.Describe(contour_line).name
    segPts = arcpy.CreateFeatureclass_management(workspace_scratch, inLineName + '_pts', 'POINT', '', '', '', sr)
    
    icursor = arcpy.da.InsertCursor(segPts, ('SHAPE@'))
    
    with arcpy.da.SearchCursor(contour_line, ('SHAPE@LENGTH', "SHAPE@")) as cursor:
        for row in cursor:
            length = row[0]
            noIntervals = int(math.floor(length / interval))
            lastSegLength = length - interval * noIntervals
            for x in range(1, noIntervals):
                newPt = row[1].positionAlongLine(interval * x)
                icursor.insertRow((newPt,))
            
            if lastSegLength < lineClusTol:
                lastPt = row[1].positionAlongLine(interval * noIntervals + lastSegLength)
            else:
                lastPt = row[1].positionAlongLine(interval * noIntervals)
            icursor.insertRow((lastPt,))
    
    del icursor  # Don't forget to close the cursor when done
except Exception as e:
    print(f"An error occurred: {str(e)}")

In [60]:
arcpy.Describe(contour_line).path

'Z:\\GIS_ArcGISPro\\TREEO\\T4T\\dem_analysis'

In [51]:
# if the sr is geographic
arcpy.env.overwriteOutput=True


outDir_temp = workspace_scratch

inFeature = os.path.join(arcpy.Describe(contour_line).path,
                         arcpy.Describe(contour_line).name)

# shapefile has the limitation, of text string length, need to convert to gdb
if arcpy.Describe(contour_line).name[-4:] == '.shp':
    arcpy.conversion.FeatureClassToGeodatabase(
        Input_Features=inFeature,
        Output_Geodatabase=r"Z:\GIS_ArcGISPro\TREEO\TREEO.gdb"
    )
    
    inFeature = os.path.join(r"Z:\GIS_ArcGISPro\TREEO\TREEO.gdb",
                         arcpy.Describe(contour_line).name[:-4])

print(inFeature)
    
interval = 3 #3 meter (spacing between trees)
lineClusTol = 1 # cluster tool, to make sure the last segment

inLineName = arcpy.Describe(inFeature).name
fields = [f.name for f in arcpy.ListFields (inFeature)]
utmField = "UTM"
i = 0
#Make sure field name is not in use

while utmField in fields:
    utmField = "UTM{0}".format(i)
    i += 1
arcpy.AddField_management (inFeature,utmField,"TEXT", field_length = 1000)

#calculate UTM zones
arcpy.CalculateUTMZone_cartography (inFeature, utmField)

#get unique value
with arcpy.da.SearchCursor(inFeature, [utmField]) as cursor:
    unique = list(set([row[0] for row in cursor]))


Z:\GIS_ArcGISPro\TREEO\TREEO.gdb\contour_line


In [53]:
print(unique,len(unique))

['PROJCS["GCS WGS 1984 UTM Zone 48M (Calculated)",GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["False_Easting",500000.0],PARAMETER["False_Northing",10000000.0],PARAMETER["Central_Meridian",105.0],PARAMETER["Scale_Factor",0.9996],PARAMETER["Latitude_Of_Origin",0.0],UNIT["Meter",1.0]]'] 1


In [54]:
unique[0]

'PROJCS["GCS WGS 1984 UTM Zone 48M (Calculated)",GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["False_Easting",500000.0],PARAMETER["False_Northing",10000000.0],PARAMETER["Central_Meridian",105.0],PARAMETER["Scale_Factor",0.9996],PARAMETER["Latitude_Of_Origin",0.0],UNIT["Meter",1.0]]'

In [55]:
sr = arcpy.SpatialReference()

In [56]:
sr.loadFromString(unique[0]) # if the data input is not shp, this will works

In [57]:
listsplitFC = []
dir = os.path.join(outDir, 'TREEO.gdb')
for i in range(len(unique)):
    unik = unique[i]
    expression =  str(utmField) + '=' + "'"+ str(unik) + "'"
    splitFC = arcpy.Select_analysis(inFeature, os.path.join(dir,inLineName+"_extracted_" + str(i)) ,expression)
    sr = arcpy.SpatialReference()
    sr.loadFromString(unik)
    splitFCProj = arcpy.Project_management(splitFC, os.path.join(outDir, inLineName+"_UTM" + str(i)) , sr)
    listsplitFC.append(splitFCProj)


In [58]:
# global listsplitFC    

#use them as input    
segPtsList = []
for i in range(len(listsplitFC)):
    inLineName = arcpy.Describe(listsplitFC[i]).name
    segPts = arcpy.CreateFeatureclass_management(outDir, inLineName + '_pts'+ str(i), 'POINT', '','','',arcpy.Describe(listsplitFC[i]).SpatialReference)
    icursor = arcpy.da.InsertCursor(segPts,('SHAPE@'))
    with arcpy.da.SearchCursor(listsplitFC[i],('SHAPE@LENGTH',"SHAPE@")) as Ucursor:
        for row in Ucursor:
             length = row[0]
             noIntervals = int(math.floor(length/interval))
             lastSegLength = length - interval * noIntervals
             for x in range(1, noIntervals):
                 newPt = row[1].positionAlongLine(interval * x)
                 icursor.insertRow((newPt,))
             if lastSegLength < lineClusTol:
                 lastPt = row[1].positionAlongLine(interval * noIntervals + lastSegLength)
             else:
                 lastPt = row[1].positionAlongLine(interval * noIntervals)
             icursor.insertRow((lastPt,))

        del Ucursor

    segPtsList.append(segPts)

In [63]:
for i in range(len(listsplitFC)):
    inLineName = arcpy.Describe(listsplitFC[i]).name
    segPts = arcpy.CreateFeatureclass_management(outDir, inLineName + '_pts' + str(i), 'POINT', '', '', '', arcpy.Describe(listsplitFC[i]).spatialReference)
    icursor = arcpy.da.InsertCursor(segPts, ('SHAPE@'))
    
    with arcpy.da.SearchCursor(listsplitFC[i], ('SHAPE@LENGTH', "SHAPE@")) as Ucursor:
        for row in Ucursor:
            i += 1
            print(f'writing to cursor {i}')
            length = row[0]
            noIntervals = int(math.floor(length / interval))
            lastSegLength = length - interval * noIntervals
            for x in range(1, noIntervals):
                newPt = row[1].positionAlongLine(interval * x)
                icursor.insertRow((newPt,))
            if lastSegLength < lineClusTol:
                lastPt = row[1].positionAlongLine(interval * noIntervals + lastSegLength)
            else:
                lastPt = row[1].positionAlongLine(interval * noIntervals)
            icursor.insertRow((lastPt,))
    
    del Ucursor
    del icursor

writing to cursor 1
writing to cursor 2
writing to cursor 3
writing to cursor 4
writing to cursor 5
writing to cursor 6
writing to cursor 7
writing to cursor 8
writing to cursor 9
writing to cursor 10
writing to cursor 11
writing to cursor 12
writing to cursor 13
writing to cursor 14
writing to cursor 15
writing to cursor 16
writing to cursor 17
writing to cursor 18
writing to cursor 19
writing to cursor 20
writing to cursor 21
writing to cursor 22
writing to cursor 23
writing to cursor 24
writing to cursor 25
writing to cursor 26
writing to cursor 27
writing to cursor 28
writing to cursor 29
writing to cursor 30
writing to cursor 31
writing to cursor 32
writing to cursor 33
writing to cursor 34
writing to cursor 35
writing to cursor 36
writing to cursor 37
writing to cursor 38
writing to cursor 39
writing to cursor 40
writing to cursor 41
writing to cursor 42
writing to cursor 43
writing to cursor 44
writing to cursor 45
writing to cursor 46
writing to cursor 47
writing to cursor 48
w

writing to cursor 750
writing to cursor 751
writing to cursor 752
writing to cursor 753
writing to cursor 754
writing to cursor 755
writing to cursor 756
writing to cursor 757
writing to cursor 758
writing to cursor 759
writing to cursor 760
writing to cursor 761
writing to cursor 762
writing to cursor 763
writing to cursor 764
writing to cursor 765
writing to cursor 766
writing to cursor 767
writing to cursor 768
writing to cursor 769
writing to cursor 770
writing to cursor 771
writing to cursor 772
writing to cursor 773
writing to cursor 774
writing to cursor 775
writing to cursor 776
writing to cursor 777
writing to cursor 778
writing to cursor 779
writing to cursor 780
writing to cursor 781
writing to cursor 782
writing to cursor 783
writing to cursor 784
writing to cursor 785
writing to cursor 786
writing to cursor 787
writing to cursor 788
writing to cursor 789
writing to cursor 790
writing to cursor 791
writing to cursor 792
writing to cursor 793
writing to cursor 794
writing to

In [62]:
arcpy.Describe(listsplitFC[0]).name

'contour_line_UTM0'

In [ ]:
#delete UTM zone field
arcpy.DeleteField_management (inFeature,utmField)

In [59]:
segPtsList

[<Result 'Z:\\GIS_ArcGISPro\\TREEO\\contour_line_UTM0_pts0.shp'>]

In [1]:
# arcpy.AddMessage("Splitting")

# for i in range(len(segPtsList)):
#     arcpy.SplitLineAtPoint_management(listsplitFC[i],segPtsList[i],os.path.join(outDir, inLineName+"_splitted" + str(i)),1)

In [16]:
# SHOULD BE IN ANOTHER NOTEBOOK
list_layers_name = [f.name for f in map.listLayers()]
list_layers_name

['class_contour_filtered_pts', 'grid_pattern_fishnet_3x4_filtered_Clip', 'point_line_per3m_contour_filtered_Clip', 'point_line_per3m_contourClip', 'grid_pattern_fishnet_3x4', 'point_line_per3m_contour', 'class_contour_filtered', 'main.contour_all_class', 'contour_line', 'prioritized_planting_filtered', 'Prioritized_planting', 'additional_t4t', 'Data_Blok_Petani_IPHPS_Mekarjaya', 'Plot_Delineated_merged', 't4t_tree_plot', 'measurement_approved_only', 'tree_space_buffered_1.5m', 'trees_buffered_gps_accuracy', 'contract_plot', 'measurement_buffered_tree_insideshp', 'Concession IPHPS - Social Forestry', 'DELINEATE_48S', 'slope_utm_fix.map', 'GEE', 'Slope_classes', 'FCD', 'Class_GEE_FCD_LC', 'PlanetLabs_used', 'T4T_202207_modified.tif', 'World Street Map']

In [17]:
point_grid = map.listLayers()[1]

In [18]:
contour_point = map.listLayers()[0]

In [19]:
# Run Near analysis to find the nearest contour_point for each point_grid
arcpy.Near_analysis(point_grid, contour_point)

<Result 'grid_pattern_fishnet_3x4_filtered_Clip'>

In [21]:
# Initialize an empty dictionary to keep track of used contour_point features
used_contour_point = {}

# Create a dictionary to map the OBJECTID of pointB to its geometry
contour_point_geometry_dict = {row[0]: row[1] for row in arcpy.da.SearchCursor(contour_point, ["FID", "SHAPE@"])}

# Iterate through point_grid features and snap them to the nearest contour_point
with arcpy.da.UpdateCursor(point_grid, ["OBJECTID","SHAPE@", "NEAR_FID"]) as cursor:
    for row in cursor:
        point_grid_geometry = row[1]
        #print(point_grid_geometry)
        near_fid = row[2]
        oid = row[0]
        
        if near_fid not in used_contour_point:
            print(f'updating oid: {oid}')
            
            geometryfrom_contour = contour_point_geometry_dict[near_fid]
            
            # Update the point_grid geometry to the nearest contour_point's location
            cursor.updateRow([oid, geometryfrom_contour, near_fid])
            
            # Mark the pointB feature as used
            used_contour_point[near_fid] = True
            
del cursor

updating oid: 1
updating oid: 3
updating oid: 5
updating oid: 7
updating oid: 9
updating oid: 10
updating oid: 11
updating oid: 12
updating oid: 15
updating oid: 16
updating oid: 22
updating oid: 31
updating oid: 33
updating oid: 36
updating oid: 37
updating oid: 38
updating oid: 39
updating oid: 41
updating oid: 43
updating oid: 47
updating oid: 48
updating oid: 50
updating oid: 52
updating oid: 55
updating oid: 56
updating oid: 57
updating oid: 59
updating oid: 62
updating oid: 63
updating oid: 64
updating oid: 65
updating oid: 68
updating oid: 70
updating oid: 72
updating oid: 74
updating oid: 75
updating oid: 77
updating oid: 79
updating oid: 81
updating oid: 84
updating oid: 86
updating oid: 88
updating oid: 89
updating oid: 90
updating oid: 95
updating oid: 101
updating oid: 105
updating oid: 106
updating oid: 107
updating oid: 109
updating oid: 111
updating oid: 112
updating oid: 114
updating oid: 116
updating oid: 118
updating oid: 120
updating oid: 122
updating oid: 124
updati

updating oid: 1867
updating oid: 1871
updating oid: 1872
updating oid: 1874
updating oid: 1875
updating oid: 1876
updating oid: 1877
updating oid: 1879
updating oid: 1880
updating oid: 1883
updating oid: 1885
updating oid: 1886
updating oid: 1889
updating oid: 1890
updating oid: 1891
updating oid: 1896
updating oid: 1902
updating oid: 1903
updating oid: 1904
updating oid: 1905
updating oid: 1906
updating oid: 1908
updating oid: 1910
updating oid: 1914
updating oid: 1915
updating oid: 1916
updating oid: 1918
updating oid: 1919
updating oid: 1920
updating oid: 1921
updating oid: 1922
updating oid: 1923
updating oid: 1927
updating oid: 1932
updating oid: 1937
updating oid: 1938
updating oid: 1939
updating oid: 1940
updating oid: 1944
updating oid: 1951
updating oid: 1954
updating oid: 1956
updating oid: 1957
updating oid: 1958
updating oid: 1959
updating oid: 1961
updating oid: 1964
updating oid: 1965
updating oid: 1968
updating oid: 1971
updating oid: 1972
updating oid: 1973
updating oid

updating oid: 3397
updating oid: 3404
updating oid: 3406
updating oid: 3407
updating oid: 3409
updating oid: 3410
updating oid: 3413
updating oid: 3414
updating oid: 3416
updating oid: 3417
updating oid: 3418
updating oid: 3420
updating oid: 3422
updating oid: 3423
updating oid: 3424
updating oid: 3425
updating oid: 3426
updating oid: 3427
updating oid: 3428
updating oid: 3429
updating oid: 3435
updating oid: 3436
updating oid: 3437
updating oid: 3440
updating oid: 3442
updating oid: 3443
updating oid: 3444
updating oid: 3445
updating oid: 3449
updating oid: 3450
updating oid: 3451
updating oid: 3453
updating oid: 3454
updating oid: 3457
updating oid: 3458
updating oid: 3459
updating oid: 3460
updating oid: 3461
updating oid: 3462
updating oid: 3466
updating oid: 3468
updating oid: 3469
updating oid: 3472
updating oid: 3473
updating oid: 3474
updating oid: 3475
updating oid: 3476
updating oid: 3478
updating oid: 3479
updating oid: 3482
updating oid: 3483
updating oid: 3488
updating oid

updating oid: 4890
updating oid: 4891
updating oid: 4892
updating oid: 4893
updating oid: 4894
updating oid: 4895
updating oid: 4896
updating oid: 4897
updating oid: 4899
updating oid: 4901
updating oid: 4902
updating oid: 4903
updating oid: 4908
updating oid: 4909
updating oid: 4911
updating oid: 4913
updating oid: 4914
updating oid: 4915
updating oid: 4916
updating oid: 4919
updating oid: 4920
updating oid: 4921
updating oid: 4922
updating oid: 4923
updating oid: 4924
updating oid: 4925
updating oid: 4927
updating oid: 4928
updating oid: 4929
updating oid: 4931
updating oid: 4932
updating oid: 4933
updating oid: 4934
updating oid: 4935
updating oid: 4936
updating oid: 4937
updating oid: 4939
updating oid: 4942
updating oid: 4943
updating oid: 4944
updating oid: 4947
updating oid: 4949
updating oid: 4950
updating oid: 4951
updating oid: 4952
updating oid: 4953
updating oid: 4954
updating oid: 4956
updating oid: 4958
updating oid: 4959
updating oid: 4960
updating oid: 4961
updating oid

In [22]:
if 8538 in used_contour_point:
    print('yes')

yes
